# **Development of Aggregation Tree Processing Tool**

In [ ]:
DP_OUTPUTS = [
    {
        "n_samples": 100,
        "model_ids": ["c1_1", "c1_2", "c1_3"]
    },
    {
        "n_samples": 120,
        "model_ids": ["c2_1", "c2_2", "c2_3"]
    }
]

AGG_TREE = {
    "processors": [
        {
            "uri": "http://processor1:8000",
            "train_data_uri": "/data/train_data_1.csv",
            "forward_model_to": [
                "http://aggregator1:8000",
                "http://aggregator2:8000",
                "http://aggregator3:8000"
            ]
        },
        {
            "uri": "http://processor2:8000",
            "train_data_uri": "/data/train_data_2.csv",
            "forward_model_to": [
                "http://aggregator1:8000",
                "http://aggregator2:8000",
                "http://aggregator3:8000"
            ]
        },
    ],
    "aggregators": [
        {
            "uri": "http://aggregator1:8000",
            "forward_model_to": [
                "http://aggregator-top:8000"
            ],
            "aggregation_level": 0
        },
        {
            "uri": "http://aggregator2:8000",
            "forward_model_to": [
                "http://aggregator-top:8000"
            ],
            "aggregation_level": 0
        },
        {
            "uri": "http://aggregator3:8000",
            "forward_model_to": [
                "http://aggregator-top:8000"
            ],
            "aggregation_level": 0
        },
        {
            "uri": "http://aggregator-top:8000",
            "forward_model_to": [],
            "aggregation_level": 1
        }
    ],
    "aggregation_algorithm": "fedavg"
}

In [ ]:
class AggregatorProcessor:
    def __init__(self, aggregation_tree):
        self.aggregation_tree = aggregation_tree

    def get_data_processor_configs(self,
                                   global_model_host,
                                   global_model_id,
                                   global_model_user,
                                   global_model_password):
        processors = []
        for _, proc in enumerate(self.aggregation_tree["processors"]):
            processors.append({
                "dp_uri": proc["uri"],
                "global_model_host": global_model_host,
                "global_model_id": global_model_id,
                "global_model_user": global_model_user,
                "global_model_password": global_model_password,
                "train_data_uri": proc["train_data_uri"],
                "forward_model_to": proc["forward_model_to"]
            })
        return processors

    def get_aggregator_configs(self,
                               dp_results,
                               global_model_host="",
                               global_model_id="",
                               global_model_user="",
                               global_model_password=""):
        #TODO: collect by aggregation level and create appropriate aggregation configs for each level
        for agg in self.aggregation_tree["aggregators"]:
                if not agg["top_level_aggregator"]:
                    params = {
                        "agg_uri": agg["uri"],
                        "model_sources": [share_results[proc["uri"]] for proc in self.aggregation_tree["processors"]],
                        "aggregation_algorithm": self.aggregation_tree["aggregation_algorithm"],
                        "forward_model_to": agg["forward_model_to"]
                    }
                    with ThreadPool(processes=2) as pool:
                        aggregates = pool.map(post_aggregate_request, [params])
                    print(aggregates)
                    agg_results = {}
                    for i in range(len(aggregates)):
                        agg_results[aggregation_tree["aggregators"][i]["uri"]] = aggregates[i]
            if aggregation_tree["aggregation_algorithm"] == "fedavg":
                # For FedAvg:
                aggregation_config = {
                    "local_model_hosts": [],
                    "local_model_uris": [],
                    "local_data_size": [],
                    "aggregation_algorithm": "fedavg"
                }
            elif aggregation_tree["aggregation_algorithm"] == "deltarep":
                # For DeltaRep:
                aggregation_config = {
                    "local_model_hosts": [],
                    "local_model_uris": [],
                    "local_data_size": [],
                    "aggregation_algorithm": "deltarep",
                    "global_model_host": global_model_host,
                    "global_model_id": global_model_id,
                    "global_model_user": global_model_user,
                    "global_model_password": global_model_password
                }
        
            for sh in shares:
                aggregation_config["local_model_hosts"].append(sh["model_server_host"])
                aggregation_config["local_model_uris"].append(sh["model_share_uri"])
                aggregation_config["local_data_size"].append(sh["n_samples"])